In [ ]:
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
import requests
import re
import io
from urllib.parse import urlencode, urlunsplit
import json

In [ ]:
response = requests.get("https://www.nexford.edu/insights/highest-paying-jobs-in-the-world")
soup = BeautifulSoup(response.content)
soup.prettify()

In [ ]:
REGEX_SALARY = re.compile(r"\$(\d+),(\d+)")
REGEX_JOBS = re.compile(r"\d{1,2}. ([A-Za-z ]*)(\\xa0)?")

trabajos_busqueda = soup.findAll(name="h3")
salarios_busqueda = soup.findAll(name="a")

salarios = []
for salario in salarios_busqueda:
    if re.search(REGEX_SALARY, salario.text):
        salarios.append(salario)
salarios = salarios[:15]

trabajos=[]
for trabajo in trabajos_busqueda:
    if re.search(REGEX_JOBS, trabajo.text):
        trabajo_regex = re.search(REGEX_JOBS, trabajo.text).group(1)
        trabajos.append(trabajo_regex)
trabajos = trabajos[:15]

In [ ]:
df = pd.DataFrame(salarios, index=trabajos, columns=["salario"])
df

In [ ]:
df = df["salario"].apply(lambda x: int(re.search(REGEX_SALARY, x).group(1)+re.search(REGEX_SALARY, x).group(2)))
df

In [ ]:
SCHEME = "https"
BASE = "jobicy.com/api/v2"
PATH = "/remote-jobs"
QUERY = {"tag":"data scientist"}

query_encoded =  urlencode(QUERY)

url=urlunsplit((SCHEME, BASE, PATH, query_encoded, ""))
response=requests.get(url)

df = pd.DataFrame.from_dict(response.json())

In [ ]:
data = []
for _, rows in df.iterrows():
    industry = rows["jobs"]["jobIndustry"]
    if "annualSalaryMin" in rows["jobs"]:
        salary = rows["jobs"]["annualSalaryMin"]
    else:
        salary = np.nan
    
    data.append(("data scientist", industry, salary))
    
df_final = pd.DataFrame(data)
df_final

In [ ]:
df_ouliers = pd.read_csv("all_jobs_3a.csv", sep=";")
df_ouliers

In [ ]:
df_ouliers["z_score_salary"] = (df_ouliers.salary - df_ouliers.salary.mean())/df_ouliers.salary.std()
median = df_ouliers["salary"].median()
df_ouliers["salary"].loc[df_ouliers["z_score_salary"].abs() > 3] = median
df_ouliers

In [ ]:
df_ouliers = df_ouliers[["job", "industry", "salary"]]


In [ ]:
df_ouliers["salary"] = df_ouliers.groupby("industry")["salary"].apply(lambda x: x.fillna(x.median())).reset_index(level=0, drop=True)
df_ouliers

In [ ]:
df_best = pd.read_csv("best_jobs_4.csv", sep=";")
df_all = pd.read_csv("all_jobs_4.csv", sep=";")

df_all


In [ ]:
df_merged = pd.merge(df_all, df_best)
df_merged["variation"] = (df_merged["salary"]-df_merged["salary_average"])/df_merged["salary"]
df_merged = df_merged[["job", "variation"]]
df_merged = df_merged.groupby("job").mean().variation.sort_values(ascending=False).reset_index()
df_merged